# pathlib & File Handling on a Real Landing Zone

## What you'll practice

`datasets/incoming/` looks like a real landing zone: data files, a corrupt file, a half-written `.tmp`, a `_SUCCESS` marker.
`datasets/daily/` has Hive-style partitions (`dt=2026-01-01/`) and one **re-delivered** file. You'll discover, inventory, partition,
archive and fingerprint files with `pathlib` — never string concatenation for paths.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/files/test_06_01_pathlib_and_files.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/06_files_json_csv/01_pathlib_and_files.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
INCOMING = DATA / "incoming"
print(sorted(p.name for p in INCOMING.iterdir()))

## Exercise 1 — 🟢 Easy · Discover input files

**Implement:** `discover_files(directory, pattern: str = '*.csv') -> list[Path]`

Return files (not directories) in `directory` matching `pattern`, **non-recursive**, sorted by name, ignoring names starting with `.` or `_` and anything ending in `.tmp`/`.part`.

**Input:** `directory, pattern: str = '*.csv'`

**Expected output:** `list[Path]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
discover_files("../datasets/incoming")   # [.../corrupt.csv, .../orders_01.csv, .../orders_02.csv, .../orders_03.csv]
```

**Check:** `pytest tests/files/test_06_01_pathlib_and_files.py -k "test_discover_files or test_discover_files_real"`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Hive-style partition paths

**Implement:** `partition_path(root, **partitions) -> Path`, `parse_partitions(path) -> dict[str, str]`

`partition_path("out", year=2026, month="01")` → `Path("out/year=2026/month=01")` (keyword order preserved). `parse_partitions` extracts every
`key=value` directory component from a path (files included): `"out/sales/dt=2026-01-05/region=IN/part-0.csv"` → `{"dt": "2026-01-05", "region": "IN"}`.

**Input:** `root, **partitions`

**Expected output:** `Path`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
parse_partitions('out/sales/dt=2026-01-05/region=IN/part-0.csv')
# -> {'dt': '2026-01-05', 'region': 'IN'}
```

**Check:** `pytest tests/files/test_06_01_pathlib_and_files.py -k "test_partition_path or test_parse_partitions"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Recursive inventory

**Implement:** `inventory(root) -> tuple[list[dict], int]`

Walk `root` recursively. Return `(files, total_bytes)` where `files` is a list of `{"path": relative POSIX path, "size": bytes, "suffix": ".csv", "partitions": {...}}`
sorted by path, and `partitions` uses your `parse_partitions` idea on the relative path. Run it on `datasets/daily`.

**Input:** `root`

**Expected output:** `tuple[list[dict], int]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_inventory, test_inventory_real_daily` in `tests/files/test_06_01_pathlib_and_files.py`.

**Check:** `pytest tests/files/test_06_01_pathlib_and_files.py -k "test_inventory or test_inventory_real_daily"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Archive processed files without clobbering

**Implement:** `archive_files(files: list, archive_root, run_id: str) -> dict[str, str]`

Move each file into `archive_root/run_id/` keeping its name. If the destination already exists, append `_1`, `_2`, ... before the suffix
(`orders.csv` → `orders_1.csv`). Missing source files → `FileNotFoundError` **before anything is moved** (validate first). Return `{src_posix: dst_posix}`.

**Input:** `files: list, archive_root, run_id: str`

**Expected output:** `dict[str, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_archive_files, test_archive_files_validates_first` in `tests/files/test_06_01_pathlib_and_files.py`.

**Check:** `pytest tests/files/test_06_01_pathlib_and_files.py -k "test_archive_files or test_archive_files_validates_first"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Detect re-delivered files by content hash · 🧩 Mini-project

**Implement:** `sha256_file(path, chunk_size: int = 65536) -> str`, `find_duplicate_files(root) -> list[list[str]]`

Upstream systems re-send files under new names. `sha256_file` hashes a file in chunks (constant memory). `find_duplicate_files(root)` returns groups
of relative POSIX paths with **identical content** (size ≥ 1 byte), each group sorted, groups sorted. Optimise: group by size first and only hash files whose
size collides. Run it on `datasets/daily` — it should find the re-delivered day-5 file in day 6.

**Input:** `path, chunk_size: int = 65536`

**Expected output:** `str`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_sha256_file, test_find_duplicate_files, test_find_duplicate_files_real` in `tests/files/test_06_01_pathlib_and_files.py`.

**Check:** `pytest tests/files/test_06_01_pathlib_and_files.py -k "test_sha256_file or test_find_duplicate_files or test_find_duplicate_files_real"`

**YOUR CODE ↓**